# Lab 2 - Joint (optional, if there's time): global meets local

Only after **both** pull requests are merged. One screen, two people.

Global told you *what usually matters*; local told you *why one hour was high*. A **dependence plot**
bridges them: it shows, feature by feature, how the SHAP value changes as the feature changes.

In [ ]:
# --- setup: install SHAP, load the data, fit the model (just run this) ---
!pip install shap -q

import pandas as pd, numpy as np, matplotlib.pyplot as plt, shap
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

url = "https://raw.githubusercontent.com/drdave-teaching/opim5512-lab2-template/main/data/energy_model_data.csv"
df = pd.read_csv(url, parse_dates=["hour"]).sort_values("hour").reset_index(drop=True)  # keep time in order

FEATURES = ["temp_f", "hour_of_day", "dewpoint_f", "humidity_pct", "wind_kt", "weekend"]
X, y = df[FEATURES], df["load_mw"]

# TIME SERIES -> no shuffle: the test set is the most recent 20% of hours (a real forecast, no peeking ahead)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, shuffle=False)
model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
print(f"model R2 (test): {r2_score(yte, model.predict(Xte)):.2f}   |   typical miss: {mean_absolute_error(yte, model.predict(Xte)):,.0f} MW")
X.head()

In [ ]:
# --- SHAP setup (just run this): explain the model on the HELD-OUT test hours ---
explainer = shap.TreeExplainer(model)
shap_values = explainer(Xte)        # explain UNSEEN data (honest) - one row per test hour, one column per feature
print("SHAP ready:", shap_values.shape, "(test hours x features)")

Run a dependence plot for the top feature. `hour_of_day` is the interesting one - watch the
evening ramp. Saves `shap_dependence.png` for the report.

In [ ]:
shap.plots.scatter(shap_values[:, "hour_of_day"], color=shap_values[:, "temp_f"])              # 1) LOOK
shap.plots.scatter(shap_values[:, "hour_of_day"], color=shap_values[:, "temp_f"], show=False)  # 2) then save
plt.gcf().savefig("shap_dependence.png", dpi=150, bbox_inches="tight"); plt.close()

from google.colab import files
files.download("shap_dependence.png")

### The Module 1 -> Module 2 payoff (write one sentence for the report)

In Lab 1 you found the hottest hour wasn't the peak-demand hour. Now the model *and* SHAP say why:
**hour-of-day carries as much weight as temperature.** The grid's evening rhythm is baked into demand,
not just the heat. Say that in plain English - it's the whole point of both labs.

## Bonus: soft-code the report (generate it from the live numbers)

The **pro move**: instead of typing numbers into `REPORT.md` by hand, *generate* it from the analysis so it can never drift or be mistyped. It links both partners' PNGs by path, so the report still assembles only once **both** merge. (If you use this, have **one** partner run it - a whole-file rewrite from both branches would collide. The hand-filled `Authors:` line is still where you practice resolving a conflict.)

In [ ]:
# generate REPORT.md from live values
top = FEATURES[int(np.argmax(np.abs(shap_values.values).mean(0)))]
base = float(shap_values.base_values[0])
r2 = r2_score(yte, model.predict(Xte))
pk = int(np.argmax(model.predict(Xte))); h = Xte.index[pk]
report = f"""# Lab 2 report - explaining our demand model

**Authors:** _replace this line with your name_

**Auto-generated stats (from the live model):**
- Top driver (mean|SHAP|): `{top}`
- Base value (average prediction): {base:,.0f} MW
- Test R2: {r2:.2f}
- Peak test hour {df.loc[h,'hour']}: predicted {model.predict(Xte)[pk]:,.0f} MW

## Global - what the model leans on overall (Partner A)
![built-in importances](images/importances_builtin.png)
![SHAP beeswarm](images/shap_global.png)

## Local - one hour explained (Partner B)
![predicted vs actual](images/predicted_vs_actual.png)
![SHAP waterfall](images/shap_local.png)

## Combined (both)
![SHAP dependence](images/shap_dependence.png)
"""
with open("REPORT.md", "w") as f:
    f.write(report)
print("wrote REPORT.md from live numbers:\n")
print(report[:420])